In [39]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

pd.set_option('display.max_columns',50)

In [54]:
teams = pd.read_csv('./data/MTeams.csv')
print("Teams data shape -->", teams.shape)
seasons = pd.read_csv('./data/MSeasons.csv')
print("Seasons data shape -->", seasons.shape)
tourney_seeds = pd.read_csv('./data/MNCAATourneySeeds.csv')
print("Tourney Seeds data shape -->", tourney_seeds.shape)
tourney_results = pd.read_csv('./data/MNCAATourneyDetailedResults.csv')
print("Tourney Results shape -->", tourney_results.shape)
detailed_regular = pd.read_csv('./data/MRegularSeasonDetailedResults.csv')
print("Detailed Regular data shape -->", detailed_regular.shape)
massey_ordinals = pd.read_csv('./data/MMasseyOrdinals.csv')
print("Massey Ordinals data shape -->", massey_ordinals.shape)

Teams data shape --> (380, 4)
Seasons data shape --> (41, 6)
Tourney Seeds data shape --> (2558, 3)
Tourney Results shape --> (1382, 34)
Detailed Regular data shape --> (118449, 34)
Massey Ordinals data shape --> (5526389, 5)


In [55]:
massey_ordinals

,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
0,2003,35,SEL,1102,159
1,2003,35,SEL,1103,229
2,2003,35,SEL,1104,12
3,2003,35,SEL,1105,314
4,2003,35,SEL,1106,260
...,...,...,...,...,...
5526384,2025,121,WOL,1476,293
5526385,2025,121,WOL,1477,347
5526386,2025,121,WOL,1478,336
5526387,2025,121,WOL,1479,302


In [56]:
massey_ordinals['SystemName'].value_counts()

SystemName
MOR    142447
POM    139956
SAG    130371
DOK    126660
MAS    116354
        ...  
PMC       351
HRN       351
CRW       351
BP5       345
PH        326
Name: count, Length: 193, dtype: int64

In [57]:
detailed_regular.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR',
       'WAst', 'WTO', 'WStl', 'WBlk', 'WPF', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3',
       'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF'],
      dtype='object')

In [58]:
tourney_results.tail()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
1377,2024,146,1301,76,1181,64,N,0,28,60,3,13,17,23,8,27,16,4,4,6,16,19,59,5,20,21,26,10,27,11,9,4,5,23
1378,2024,146,1345,72,1397,66,N,0,24,53,3,15,21,33,8,32,16,10,5,2,12,24,62,11,26,7,11,6,17,17,6,8,4,25
1379,2024,152,1163,86,1104,72,N,0,31,62,10,25,14,18,10,25,20,4,4,8,17,26,58,11,23,9,11,7,21,9,7,2,5,15
1380,2024,152,1345,63,1301,50,N,0,22,55,10,25,9,10,10,28,13,14,5,2,8,21,57,5,19,3,4,6,22,10,11,8,3,13
1381,2024,154,1163,75,1345,60,N,0,30,62,6,22,9,11,13,19,18,6,3,4,18,24,54,1,7,11,15,8,19,8,9,3,3,15


In [59]:
detailed_regular.tail()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,WFGM3,WFGA3,WFTM,WFTA,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LFGM,LFGA,LFGM3,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
118444,2025,120,1433,71,1182,62,A,0,23,55,8,22,17,21,12,22,10,14,9,4,23,21,51,4,16,16,24,11,19,9,15,8,5,16
118445,2025,120,1436,79,1107,71,H,0,28,48,11,22,12,15,4,23,14,12,2,9,11,25,61,9,20,12,15,10,18,11,8,9,0,15
118446,2025,120,1438,60,1199,57,H,0,21,53,11,24,7,9,6,20,15,11,3,7,12,23,62,7,22,4,7,12,21,12,8,7,9,15
118447,2025,120,1452,71,1428,69,A,0,26,57,8,23,11,16,6,21,14,10,7,2,20,19,50,9,28,22,32,10,26,11,16,7,3,17
118448,2025,120,1460,98,1237,85,H,0,36,57,14,30,12,15,10,24,27,11,2,4,11,31,58,14,24,9,14,4,12,16,4,7,1,16


In [60]:
detailed_regular['WFG2Perc'] = (detailed_regular['WFGM']-detailed_regular['WFGM3'])/(detailed_regular['WFGA']-detailed_regular['WFGA3'])
detailed_regular['LFG2Perc'] = (detailed_regular['LFGM']-detailed_regular['LFGM3'])/(detailed_regular['LFGA']-detailed_regular['LFGA3'])
detailed_regular['WFG3Perc'] = detailed_regular['WFGM3']/detailed_regular['WFGA3']
detailed_regular['LFG3Perc'] = detailed_regular['LFGM3']/detailed_regular['LFGA3']
detailed_regular['WFTPerc'] = detailed_regular['WFTM']/(detailed_regular['WFTA']+1)
detailed_regular['LFTPerc'] = detailed_regular['LFTM']/(detailed_regular['LFTA']+1)

In [61]:
drop_cols = ['WTeamID','LTeamID','WLoc','NumOT','WFGM','WFGA','LFGM','LFGA','WFGM3','WFGA3','LFGM3','LFGA3','WFTM','WFTA','LFTM','LFTA']
detailed_regular.drop(drop_cols, inplace=True, axis=1)
detailed_regular

,Season,DayNum,WScore,LScore,WOR,WDR,WAst,WTO,WStl,WBlk,WPF,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WFG2Perc,LFG2Perc,WFG3Perc,LFG3Perc,WFTPerc,LFTPerc
0,2003,10,68,62,14,24,13,23,7,1,22,10,22,8,18,9,2,20,0.545455,0.465116,0.214286,0.200000,0.578947,0.695652
1,2003,10,70,63,15,28,16,13,4,4,18,20,25,7,12,8,6,16,0.428571,0.418605,0.400000,0.250000,0.500000,0.428571
2,2003,11,73,61,17,26,15,10,5,2,25,31,22,9,12,2,5,23,0.400000,0.404255,0.444444,0.115385,0.566667,0.583333
3,2003,11,56,50,6,19,11,12,14,2,18,17,20,9,19,4,3,23,0.517241,0.444444,0.333333,0.272727,0.531250,0.500000
4,2003,11,77,71,17,22,12,14,4,4,20,21,15,12,10,7,1,14,0.510638,0.391304,0.428571,0.375000,0.785714,0.607143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118444,2025,120,71,62,12,22,10,14,9,4,23,11,19,9,15,8,5,16,0.454545,0.485714,0.363636,0.250000,0.772727,0.640000
118445,2025,120,79,71,4,23,14,12,2,9,11,10,18,11,8,9,0,15,0.653846,0.390244,0.500000,0.450000,0.750000,0.750000
118446,2025,120,60,57,6,20,15,11,3,7,12,12,21,12,8,7,9,15,0.344828,0.400000,0.458333,0.318182,0.700000,0.500000
118447,2025,120,71,69,6,21,14,10,7,2,20,10,26,11,16,7,3,17,0.529412,0.454545,0.347826,0.321429,0.647059,0.666667


In [62]:
detailed_regular.columns

Index(['Season', 'DayNum', 'WScore', 'LScore', 'WOR', 'WDR', 'WAst', 'WTO',
       'WStl', 'WBlk', 'WPF', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk',
       'LPF', 'WFG2Perc', 'LFG2Perc', 'WFG3Perc', 'LFG3Perc', 'WFTPerc',
       'LFTPerc'],
      dtype='object')

In [63]:
def assign_teams(df):
    # true means winner goes to Team A
    coin = np.random.rand(len(df)) < 0.5

    # team A stats based on coin toss
    teamA = pd.DataFrame({
        'Season': df['Season'],
        'DayNum': df['DayNum'],
        'Score': np.where(coin, df['WScore'], df['LScore']),
        'OR': np.where(coin, df['WOR'], df['LOR']),
        'DR': np.where(coin, df['WDR'], df['LDR']),
        'Ast': np.where(coin, df['WAst'], df['LAst']),
        'TO': np.where(coin, df['WTO'], df['LTO']),
        'Stl': np.where(coin, df['WStl'], df['LStl']),
        'Blk': np.where(coin, df['WBlk'], df['LBlk']),
        'PF': np.where(coin, df['WPF'], df['LPF']),
        'FG2Perc': np.where(coin, df['WFG2Perc'], df['LFG2Perc']),
        'FG3Perc': np.where(coin, df['WFG3Perc'], df['LFG3Perc']),
        'FTPerc': np.where(coin, df['WFTPerc'], df['LFTPerc']),
    })

    # team B stats based on coin toss
    teamB = pd.DataFrame({
        'Score': np.where(coin, df['LScore'], df['WScore']),
        'OR': np.where(coin, df['LOR'], df['WOR']),
        'DR': np.where(coin, df['LDR'], df['WDR']),
        'Ast': np.where(coin, df['LAst'], df['WAst']),
        'TO': np.where(coin, df['LTO'], df['WTO']),
        'Stl': np.where(coin, df['LStl'], df['WStl']),
        'Blk': np.where(coin, df['LBlk'], df['WBlk']),
        'PF': np.where(coin, df['LPF'], df['WPF']),
        'FG2Perc': np.where(coin, df['LFG2Perc'], df['WFG2Perc']),
        'FG3Perc': np.where(coin, df['LFG3Perc'], df['WFG3Perc']),
        'FTPerc': np.where(coin, df['LFTPerc'], df['WFTPerc']),
    })

    # Create Win column: 1 if Team A gets the winner's stats, 0 otherwise
    win = pd.Series(np.where(coin, 1, 0), name='Win')

    # Optionally, rename columns to clearly indicate Team A and B stats (except for Season)
    teamA = teamA.rename(columns=lambda x: x if x=='Season' else x + '_A')
    teamB = teamB.rename(columns=lambda x: x if x=='Season' else x + '_B')

    # Concatenate the two teams' stats and the Win column into one DataFrame
    new_df = pd.concat([teamA, teamB, win], axis=1)
    
    return new_df

new_detailed_reg = assign_teams(detailed_regular)
new_detailed_reg.head()

,Season,DayNum_A,Score_A,OR_A,DR_A,Ast_A,TO_A,Stl_A,Blk_A,PF_A,FG2Perc_A,FG3Perc_A,FTPerc_A,Score_B,OR_B,DR_B,Ast_B,TO_B,Stl_B,Blk_B,PF_B,FG2Perc_B,FG3Perc_B,FTPerc_B,Win
0,2003,10,62,10,22,8,18,9,2,20,0.465116,0.200000,0.695652,68,14,24,13,23,7,1,22,0.545455,0.214286,0.578947,0
1,2003,10,63,20,25,7,12,8,6,16,0.418605,0.250000,0.428571,70,15,28,16,13,4,4,18,0.428571,0.400000,0.500000,0
2,2003,11,61,31,22,9,12,2,5,23,0.404255,0.115385,0.583333,73,17,26,15,10,5,2,25,0.400000,0.444444,0.566667,0
3,2003,11,50,17,20,9,19,4,3,23,0.444444,0.272727,0.500000,56,6,19,11,12,14,2,18,0.517241,0.333333,0.531250,0
4,2003,11,77,17,22,12,14,4,4,20,0.510638,0.428571,0.785714,71,21,15,12,10,7,1,14,0.391304,0.375000,0.607143,1


In [64]:
new_detailed_reg['Win'].value_counts()

Win
1    59248
0    59201
Name: count, dtype: int64

In [ ]:
new_detailed_reg['PM'] = new_detailed_reg['Score_A'] - new_detailed_reg['Score_B']
new_detailed_reg['OR_diff'] = new_detailed_reg['OR_A'] - new_detailed_reg['OR_B']
new_detailed_reg['DR_diff'] = new_detailed_reg['DR_A'] - new_detailed_reg['DR_B']